# S4b.4 — Parent-Child Chunk Retrieval

**Spec**: Two-tier chunking strategy — index small child chunks (200w) for precise retrieval, expand to parent chunks (600w) for richer LLM context.

**Key Classes**: `ParentChildChunker`, `ChildChunk`, `ParentChildResult`

In [1]:
import sys
sys.path.insert(0, "../..")

from src.services.indexing.parent_child import ParentChildChunker
from src.schemas.indexing import ChildChunk, ParentChildResult, TextChunk

print("\u2713 Imports successful")

✓ Imports successful


## 1. Initialization & Validation

In [2]:
# Default config
chunker = ParentChildChunker()
assert chunker.parent_chunk_size == 600
assert chunker.child_chunk_size == 200
assert chunker.child_overlap == 50
print(f"Default: parent={chunker.parent_chunk_size}, child={chunker.child_chunk_size}, overlap={chunker.child_overlap}")

# Custom config
custom = ParentChildChunker(parent_chunk_size=800, child_chunk_size=250, child_overlap=60)
assert custom.parent_chunk_size == 800
print(f"Custom: parent={custom.parent_chunk_size}, child={custom.child_chunk_size}, overlap={custom.child_overlap}")

# Validation
try:
    ParentChildChunker(parent_chunk_size=200, child_chunk_size=300)
    assert False, "Should have raised ValueError"
except ValueError as e:
    print(f"\u2713 Validation works: {e}")

print("\n\u2713 Initialization & validation passed")

Default: parent=600, child=200, overlap=50
Custom: parent=800, child=250, overlap=60
✓ Validation works: child_chunk_size must be less than parent_chunk_size

✓ Initialization & validation passed


## 2. Create Parent-Child Chunks from Paper Text

In [3]:
# Generate sample paper text (~1500 words)
sample_text = " ".join([f"word{i}" for i in range(1500)])

chunker = ParentChildChunker(parent_chunk_size=600, child_chunk_size=200, child_overlap=50)
result = chunker.create_parent_child_chunks(
    title="Attention Is All You Need",
    abstract="We propose a new architecture based on attention mechanisms.",
    full_text=sample_text,
    arxiv_id="1706.03762",
    paper_id="uuid-test-123",
)

assert isinstance(result, ParentChildResult)
assert len(result.parents) > 0
assert len(result.children) > len(result.parents)

print(f"Parents: {len(result.parents)}")
print(f"Children: {len(result.children)}")
print(f"Ratio: {len(result.children) / len(result.parents):.1f}x more children than parents")

# Verify all children reference valid parents
parent_indices = {p.metadata.chunk_index for p in result.parents}
for child in result.children:
    assert child.parent_chunk_index in parent_indices

print("\n\u2713 All children correctly reference their parents")

Parents: 3
Children: 11
Ratio: 3.7x more children than parents

✓ All children correctly reference their parents


## 3. Split Single Parent into Children

In [4]:
from src.schemas.indexing import ChunkMetadata

parent = result.parents[0]
children = chunker.split_parent_into_children(parent)

print(f"Parent word count: {parent.metadata.word_count}")
print(f"Number of children: {len(children)}")
for i, child in enumerate(children):
    print(f"  Child {i}: {child.metadata.word_count} words, parent_idx={child.parent_chunk_index}")

# Verify sequential indices
indices = [c.metadata.chunk_index for c in children]
assert indices == list(range(len(children)))

print("\n\u2713 Parent split into children correctly")

Parent word count: 600
Number of children: 4
  Child 0: 200 words, parent_idx=0
  Child 1: 200 words, parent_idx=0
  Child 2: 200 words, parent_idx=0
  Child 3: 150 words, parent_idx=0

✓ Parent split into children correctly


## 4. Section-Based Parent-Child Chunking

In [5]:
sections = {
    "Introduction": " ".join([f"intro{i}" for i in range(400)]),
    "Methods": " ".join([f"method{i}" for i in range(500)]),
    "Results": " ".join([f"result{i}" for i in range(300)]),
}

result_sec = chunker.create_parent_child_chunks(
    title="Test Paper with Sections",
    abstract="A paper about testing.",
    full_text=sample_text,
    arxiv_id="2401.12345",
    paper_id="uuid-sec-456",
    sections=sections,
)

print(f"Section-based: {len(result_sec.parents)} parents, {len(result_sec.children)} children")
for p in result_sec.parents:
    print(f"  Parent {p.metadata.chunk_index}: section='{p.metadata.section_title}', words={p.metadata.word_count}")

print("\n\u2713 Section-based parent-child chunking works")

Section-based: 3 parents, 9 children
  Parent 0: section='Introduction', words=411
  Parent 1: section='Methods', words=511
  Parent 2: section='Results', words=311

✓ Section-based parent-child chunking works


## 5. Prepare for OpenSearch Indexing

In [6]:
docs = chunker.prepare_for_indexing(result)

print(f"Prepared {len(docs)} documents for indexing")
print(f"\nSample document:")
sample_doc = docs[0]["chunk_data"]
for key, value in sample_doc.items():
    display_val = f"{str(value)[:80]}..." if len(str(value)) > 80 else value
    print(f"  {key}: {display_val}")

# Verify parent_chunk_id format
for doc in docs:
    pid = doc["chunk_data"]["parent_chunk_id"]
    assert pid.startswith("1706.03762_parent_")

print("\n\u2713 All documents have valid parent_chunk_id")

Prepared 11 documents for indexing

Sample document:
  chunk_text: word0 word1 word2 word3 word4 word5 word6 word7 word8 word9 word10 word11 word12...
  arxiv_id: 1706.03762
  paper_id: uuid-test-123
  chunk_index: 0
  chunk_word_count: 200
  start_char: 0
  end_char: 1489
  parent_chunk_id: 1706.03762_parent_0
  section_title: None

✓ All documents have valid parent_chunk_id


## 6. Parent Expansion (Retrieval-Time)

In [7]:
from unittest.mock import MagicMock

# Simulate child search results
child_results = [
    {"chunk_text": "child from parent 0", "parent_chunk_id": "1706.03762_parent_0", "score": 0.95},
    {"chunk_text": "another child from parent 0", "parent_chunk_id": "1706.03762_parent_0", "score": 0.80},
    {"chunk_text": "child from parent 1", "parent_chunk_id": "1706.03762_parent_1", "score": 0.75},
]

# Mock OpenSearch client
os_client = MagicMock()
os_client.client.get.side_effect = [
    {"_source": {"chunk_text": "Full parent 0 text (600 words of context)", "chunk_index": 0}},
    {"_source": {"chunk_text": "Full parent 1 text (600 words of context)", "chunk_index": 1}},
]

parents = chunker.expand_to_parents(child_results, os_client)

# 3 children from 2 parents → 2 unique parents
assert len(parents) == 2
print(f"3 child results → {len(parents)} unique parents (deduplicated)")
print(f"Best score preserved: {parents[0]['score']}")

print("\n\u2713 Parent expansion with deduplication works")

3 child results → 2 unique parents (deduplicated)
Best score preserved: 0.95

✓ Parent expansion with deduplication works


## Summary

S4b.4 implements a two-tier chunking strategy:
- **Parent chunks** (600w): Rich context for LLM generation
- **Child chunks** (200w, 50 overlap): Precise retrieval targets
- **At query time**: Child matches → expand to parent chunks → deduplicate → feed to LLM

This improves retrieval precision while maintaining generation quality.